# Simulation: Phish to containment

A reported phishing email, a ticking clock and a laptop running malware. Find who opened it, contain it before 09:41, and hand it on.

The files for this seminar are in `~/case` (Linux) and `case/` (Python): `alerts-fs01.json`, `alerts.json`, `eventlogs/Security.jsonl`, `invoice.eml`, `mail.log`, `process.csv`, `proxy.log`, `README.txt`, `share_listing.txt`, `ticket-88213.txt`.

## 1. The report

It's Monday 2 November 2026 at **Kestrel Logistics**, 09:16 UTC. Dev Kapoor in Finance has just reported an email as phishing: message **4A1F21C02**, an overdue invoice. Read it, then the mail server's log, to find out who else got it.

- [ ] Who really sent it? Read the From and Return-Path, and the domain, letter by letter
- [ ] How many people received message 4A1F21C02, and who?

> Hint: In mail.log, `nrcpt=` is the number of recipients, and each `to=<...>` line is one mailbox.

?> Which domain sent the invoice? :: sha256:a12c8fe90e1a9758a4bd25fa2da95ffba7ae75dc819e74dbddd34815c27e9e6a
?> How many people received message 4A1F21C02? :: sha256:4e07408562bedb8b60ce05c1decfe3ad16b72230967de01f640b7e4729b49fce

In [ ]:
%%bash
cd ~/case
grep -E '^(From|Return-Path|Subject):' invoice.eml
echo ---
grep 4A1F21C02 mail.log

## 2. Who opened it

Receiving a phishing email is not an incident. **Opening** it is. The web proxy logs every download. Search it for the look-alike domain, and turn the epoch times into UTC.

- [ ] Which internal address downloaded INV-30412.docm, and when?
- [ ] Did anyone else try? Did it work?

> Hint: `TCP_MISS/200` means the file was fetched. `TCP_DENIED/403` means the proxy blocked it: the domain was added to the blocklist at 09:18.

?> Which internal address downloaded INV-30412.docm? :: sha256:ecda40e9a719dbe5a19ca890028508dc0dffab4ef41d0accd82d2c3a46a8405d
?> Did fin-pc02 get the document? (yes or no) :: sha256:9390298f3fb0c5b160498935d79cb139aef28e1c47358b4bbba61862b9c26e59

In [ ]:
%%bash
cd ~/case
grep logistlcs proxy.log
echo ---
grep logistlcs proxy.log | while read t ms ip code size method url rest; do echo "$(date -u -d @${t%.*} '+%F %T') $ip $code $url"; done

## 3. What ran on hr-pc07

10.20.0.14 is hr-pc07, Priya Nair's HR laptop. The EDR process list shows every program that started and which program started it. Follow the chain from Word.

- [ ] What did Word start?
- [ ] What did that start, and what was left running?

> Hint: Follow the `ppid` column: each process's parent is the row whose `pid` matches it.

?> Which program did Word start? :: sha256:7371f071a9a4e653a5afd134bce9c735ef74b0421d6988958e5c6d8a34feaa3b
?> Which program was left running from C:\Users\Public? :: sha256:3753b9feba052dd133333c16e62f6adeeb6985f2999fc5d97ab190d85024f506

In [ ]:
%%bash
cd ~/case
grep hr-pc07 process.csv

## 4. Contain, then escalate

You know enough: hr-pc07 opened a macro document, and malware is running on it. **Contain first, investigate second.** In the Hacker Room, isolate hr-pc07 from the network (Remote desktop, or shut its switch port in the Server room), end the malware's process, and escalate alert AL-2203 with a note that says what you found and what you did.

If nobody contains it, at 09:41 the malware walks the file server and renames the HR share's files to `.kestrel-locked`. `share_listing.txt` shows what that looks like.

- [ ] Isolate hr-pc07 before 09:41
- [ ] End svchost-upd.exe
- [ ] Escalate AL-2203: what fired, what you found, what you did, what you don't know yet

> Hint: Don't switch the laptop off: memory holds evidence. Isolating keeps it running but cut off.

?> Which alert should you escalate? (its ID) :: sha256:48b38624c974f2a444d1d554620bcc29254c30fe3ff7bdc4079e421da59ba289